In [ ]:
from pyspark import SparkConf, SparkContext
from pyspark.mllib.tree import DecisionTree, DecisionTreeModel
from pyspark.mllib.util import MLUtils
"""--------------------------------------------------------------------------
RDD-based decision tree learning
-------------------------------------------------------------------------"""
# Create a SparkContext
conf = SparkConf().setMaster("local").setAppName("DecisionTreeCancerExample") #treat every core of your desktop as an executor
SpContext = SparkContext(conf = conf)

# Download the Marvel graph and names data files
!wget -O 8-breast-cancer.txt https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-breast-cancer.txt
# Load and parse the data file (in SVM format ) into an RDD of LabeledPoint.
data = MLUtils.loadLibSVMFile(SpContext, '8-breast-cancer.txt')
# Split the data into training and test sets (20% held out for testing)
(trainingDataSet, testDataSet) = data.randomSplit([0.8, 0.2])
# Train a DecisionTree model.
# Empty categoricalFeaturesInfo indicates all features are continuous.
model = DecisionTree.trainClassifier(trainingDataSet, impurity='gini', \
numClasses=2, maxDepth=3, maxBins=100,
categoricalFeaturesInfo={})
#print the learned model
print('Learned classification tree model:')
print(str(model.toDebugString()))
# Evaluate model on test instances and compute test error
predictions = model.predict(testDataSet.map(lambda testData: testData.features))
labelsAndPredictions = testDataSet.map(lambda testData:
testData.label).zip(predictions)
testErr = labelsAndPredictions.filter(lambda item: item[0] != item[1]).count() /\
float(testDataSet.count())
print('Test Error Rate = ' + str(testErr))
print('Test Accuracy = '+ str(1-testErr))
# Save and load model example code
#model.save(SpContext,"myDecisionTreeClassificationModel")
#sameModel = DecisionTree

--2026-03-01 19:08:00--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-breast-cancer.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 82965 (81K) [text/plain]
Saving to: ‘8-breast-cancer.txt’

8-breast-cancer.txt 100%[===================>]  81.02K  --.-KB/s    in 0.002s  

2026-03-01 19:08:00 (41.3 MB/s) - ‘8-breast-cancer.txt’ saved [82965/82965]

Learned classification tree model:
DecisionTreeModel classifier of depth 3 with 11 nodes
  If (feature 2 <= 2.5)
   If (feature 6 <= 5.5)
    If (feature 1 <= 7.5)
     Predict: 0.0
    Else (feature 1 > 7.5)
     Predict: 1.0
   Else (feature 6 > 5.5)
    Predict: 1.0
  Else (feature 2 > 2.5)
   If (feature 3 <= 2.5)
    If (feature 1 <= 5.5)
     Predict: 0.0
    Else (feature 1 > 5.5)


In [ ]:
# -*- coding: utf-8 -*-
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.mllib.util import MLUtils
"""--------------------------------------------------------------------------
dataframe-based decision tree learning
-------------------------------------------------------------------------"""
print("=== starting dataframe-based decision tree learning ===")
# Create a SparkContext
conf = SparkConf().setMaster("local").setAppName("DecisionTreeBreastCancer")
# Stop any existing SparkContext if it's running
if 'SpContext' in globals() and SpContext:
    SpContext.stop()
SpContext = SparkContext(conf = conf)
# Create a SparkSession (the config bit is only for Windows!)
SpSession = SparkSession.builder.getOrCreate()

"""--------------------------------------------------------------------------
Load Data
-------------------------------------------------------------------------"""
print("=== start loading data ===")
#Load the CSV file into a RDD
breastcancerData = SpContext.textFile("8-breast-cancer.txt")
breastcancerData.cache()

"""--------------------------------------------------------------------------
Cleanup and read data into mamory
-------------------------------------------------------------------------"""
print("=== start reading data ===")
from pyspark.sql import Row

def parse_libsvm_line_to_row(line):
    parts = line.strip().split(" ")
    label = float(parts[0])
    features_dict = {}
    for feature_str in parts[1:]:
        if ":" in feature_str:
            idx, val = feature_str.split(":")
            features_dict[int(idx)] = float(val)

    # Assuming 10 features, 1-indexed, and mapping them to named columns
    # based on the original code's Row structure.
    # It seems the original code was mapping p[1] to RADIUS (feature 1), p[2] to TEXTURE (feature 2), etc.
    # So, we expect features indexed 1 through 10.

    return Row(
        OUTCOME=label,
        RADIUS=features_dict.get(1, 0.0),
        TEXTURE=features_dict.get(2, 0.0),
        PERIMETER_AREA=features_dict.get(3, 0.0),
        SMOOTHNESS=features_dict.get(4, 0.0),
        COMPACTNESS=features_dict.get(5, 0.0),
        CONCAVITY=features_dict.get(6, 0.0),
        CONCAVE_POINTS=features_dict.get(7, 0.0),
        FRACTAL_DIMENSION=features_dict.get(8, 0.0),
        SYMMETRY=features_dict.get(9, 0.0),
        DIMENTION=features_dict.get(10, 0.0)
    )

#Create a Data Frame from the data
breastcancerMap = breastcancerData.map(parse_libsvm_line_to_row) # Use the new parsing function
# Infer the schema, and register the DataFrame as a table.
breastcancerDf = SpSession.createDataFrame(breastcancerMap)
breastcancerDf.cache()
#Add a numeric indexer for the label/target column
#Why? because the algorithm cannot handle string names
from pyspark.ml.feature import StringIndexer
stringIndexer = StringIndexer(inputCol="OUTCOME", outputCol="IND_OUTCOME")
si_model = stringIndexer.fit(breastcancerDf)
breastcancerNormDf = si_model.transform(breastcancerDf)
breastcancerNormDf.select("OUTCOME", "IND_OUTCOME").distinct().collect()
breastcancerNormDf.cache()
breastcancerNormDf.describe().show(10)
"""--------------------------------------------------------------------------
Perform Data Analytics
-------------------------------------------------------------------------"""
print("=== start data analysis ===")
#Find correlation between predictors and target
for i in breastcancerNormDf.columns:
    if not( isinstance(breastcancerNormDf.select(i).take(1)[0][0], str)) :
        print( "Correlation to OUTCOME for ", i, \
               breastcancerNormDf.stat.corr('IND_OUTCOME', i))
"""--------------------------------------------------------------------------
Prepare data for ML
-------------------------------------------------------------------------"""
print("=== start prepare data for ML ===")
#Transform to a Data Frame for input to Machine Learing
#Drop columns that are not required (low correlation)
from pyspark.ml.linalg import Vectors
def transformToLabeledPoint(row) :
    lp = ( row["OUTCOME"], row["IND_OUTCOME"],
           Vectors.dense([row["RADIUS"],
                          row["TEXTURE"],
                          row["PERIMETER_AREA"],
                          row["SMOOTHNESS"],
                          row["COMPACTNESS"],
                          row["CONCAVITY"],
                          row["CONCAVE_POINTS"],
                          row["FRACTAL_DIMENSION"],
                          row["SYMMETRY"]]))
    return lp
breastcancerLp = breastcancerNormDf.rdd.map(transformToLabeledPoint)
breastcancerLpDf = SpSession.createDataFrame(breastcancerLp, ["outcome", "label",
"features"])
breastcancerLpDf.select("outcome", "label", "features").orderBy("features").show()
breastcancerLpDf.cache()
"""--------------------------------------------------------------------------
Perform Machine Learning
-------------------------------------------------------------------------"""
print("=== start building model ===")
#Split into training and testing data
(trainingData, testData) = breastcancerLpDf.randomSplit([0.8, 0.2])
trainingData.count()
testData.count()
testData.collect()
#import the decision tree algorithm class and its evaluator
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
#Create the model
dtClassifer = DecisionTreeClassifier(maxDepth=2,
labelCol="label",featuresCol="features")
dtModel = dtClassifer.fit(trainingData)
"""--------------------------------------------------------------------------
Print out learned model
-------------------------------------------------------------------------"""
print("=== print out learned model ===")
nodeCount = dtModel.numNodes
depth = dtModel.depth
treeRules = dtModel
print("nodeCount: "+str(nodeCount)) #show how many nodes were learned
print("depth: "+str(depth)) #show the depth of the tree
print(str(treeRules.toDebugString)) #show the learned tree as rules
"""--------------------------------------------------------------------------
Evaluate learned model
-------------------------------------------------------------------------"""
print("=== print evaluation reults ===")
#Predict on the test data
predictions = dtModel.transform(testData)
predictions.select("prediction","outcome","label").collect()
#Evaluate precision
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction", \
labelCol="label",metricName="weightedPrecision")
precision = evaluator.evaluate(predictions)
print("The precision of the model is: "+str(precision))
print("The error rate of the model is: "+str(1-precision))
print("\n=== print confusion matrix ===")
#Draw a confusion matrix
predictions.groupBy("label","prediction").count().show()

=== starting dataframe-based decision tree learning ===
=== start loading data ===
=== start reading data ===
+-------+-------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+-------------------+
|summary|            OUTCOME|            RADIUS|           TEXTURE|    PERIMETER_AREA|        SMOOTHNESS|      COMPACTNESS|         CONCAVITY|    CONCAVE_POINTS|FRACTAL_DIMENSION|          SYMMETRY|         DIMENTION|        IND_OUTCOME|
+-------+-------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+-------------------+
|  count|                683|               683|               683|               683|               683|              683|               683|               683|              6

In [ ]:
# Random Forest
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import Row
"""--------------------------------------------------------------------------
Create spark context and spark session
-------------------------------------------------------------------------"""
# Create a SparkContext
# Stop any existing SparkContext if it's running
if 'SpContext' in globals() and SpContext:
    SpContext.stop()

conf = SparkConf().setMaster("local").setAppName("RandomForestExample") # treat every core of your desktop as an executor
SpContext = SparkContext(conf = conf)
# Create a SparkSession (the config bit is only for Windows!)
SpSession = SparkSession.builder.config("spark.sql.warehouse.dir",
"file:///C:/temp").getOrCreate()

"""--------------------------------------------------------------------------
Load Data
-------------------------------------------------------------------------"""
print("=== start loading data ===")
# Download the Marvel graph and names data files
!wget -O 9-banking-data.csv https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/9-banking-data.csv

#Load the CSV file into a RDD
bankData = SpContext.textFile("9-banking-data.csv")
bankData.cache()
bankData.count()
#Remove the first line using filter (contains headers)
firstLine=bankData.first() #get the first line
dataLines = bankData.filter(lambda x: x != firstLine) #filter by checking if it is the first line
dataLines.count()
"""--------------------------------------------------------------------------
Cleanup Data
-------------------------------------------------------------------------"""
print("=== start clearning data ===")
# Change labels to numeric ones and build a Row object
import math
from pyspark.ml.linalg import Vectors
def transformToNumeric( inputStr) :
    attList=inputStr.replace("\"","").split(";") #revmoving all " sign, split data with ;
    age=float(attList[0])
    #convert outcome label to float
    outcome = 0.0 if attList[16] == "no" else 1.0
    #create binary indicator variables for single/married
    single= 1.0 if attList[2] == "single" else 0.0
    married = 1.0 if attList[2] == "married" else 0.0
    divorced = 1.0 if attList[2] == "divorced" else 0.0
    #create indicator variables for education
    primary = 1.0 if attList[3] == "primary" else 0.0
    secondary = 1.0 if attList[3] == "secondary" else 0.0
    tertiary = 1.0 if attList[3] == "tertiary" else 0.0
    #convert loan default flag to float
    default= 0.0 if attList[4] == "no" else 1.0
    #convert balance amount to float
    balance=float(attList[5])
    #convert loan to float
    loan= 0.0 if attList[7] == "no" else 1.0
    #Create a row with cleaned up and converted data
    values= Row( OUTCOME=outcome ,\
    AGE=age, \
    SINGLE=single, \
    MARRIED=married, \
    DIVORCED=divorced, \
    PRIMARY=primary, \
    SECONDARY=secondary, \
    TERTIARY=tertiary, \
    DEFAULT=default, \
    BALANCE=balance, \
    LOAN=loan
    )
    return values

#Change to a Vector
bankRows = dataLines.map(transformToNumeric)
bankRows.collect()
bankData = SpSession.createDataFrame(bankRows)
"""--------------------------------------------------------------------------
Perform Data Analytics
-------------------------------------------------------------------------"""
#See descriptive statistical analytics, e.g. mean, count, max, min.
print("=== start basic statistical analysis on features ===")
bankData.describe().show()
#Find correlation between predictors and target
for i in bankData.columns:
    if not( isinstance(bankData.select(i).take(1)[0][0], str)) :
      print( "Correlation to OUTCOME for ", i, \
      bankData.stat.corr('OUTCOME',i))
"""--------------------------------------------------------------------------
Prepare data for ML
-------------------------------------------------------------------------"""
#Transform to a Data Frame for input to Machine Learing
print("=== create labeled points and ML features ===")
def transformToLabeledPoint(row) : #tranform rows to labled point
    lp = (
        row["OUTCOME"], \
          Vectors.dense(
              [
                  row["AGE"], \
                  row["BALANCE"], \
                  row["DEFAULT"], \
                  row["DIVORCED"], \
                  row["LOAN"], \
                  row["MARRIED"], \
                  row["PRIMARY"], \
                  row["SECONDARY"], \
                  row["SINGLE"], \
                  row["TERTIARY"]
              ]
         )
    )
    return lp

bankLp = bankData.rdd.map(transformToLabeledPoint)
bankLp.collect()
bankDF = SpSession.createDataFrame(bankLp,["label", "features"])
bankDF.select("label","features").show(10)
# print('\n')
"""--------------------------------------------------------------------------
Perform Machine Learning
-------------------------------------------------------------------------"""
print("=== create PCA as ML features ===")
#Perform PCA
from pyspark.ml.feature import PCA
bankPCA = PCA(k=4, inputCol="features", outputCol="pcaFeatures")
pcaModel = bankPCA.fit(bankDF)
pcaResult = pcaModel.transform(bankDF).select("label","pcaFeatures")
pcaResult.show(truncate=False)
#Indexing needed as pre-req
from pyspark.ml.feature import StringIndexer
stringIndexer = StringIndexer(inputCol="label", outputCol="indexed")
si_model = stringIndexer.fit(pcaResult)
td = si_model.transform(pcaResult)
td.collect()
#Split into training and testing data
(trainingData, testData) = td.randomSplit([0.8, 0.2])
trainingData.count()
testData.count()
testData.collect()

# Random Forest
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
print("=== use PCA features to build models ===")
#Create the model
rmClassifer = RandomForestClassifier(labelCol="indexed", \
featuresCol="pcaFeatures") # unless the seed is set, the decision tree can be changing, which means result can be changing every run.
rmModel = rmClassifer.fit(trainingData)
#Predict on the test data
predictions = rmModel.transform(testData)
predictions.select("prediction","indexed","label","pcaFeatures").collect()
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction", \
labelCol="indexed",metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
"""--------------------------------------------------------------------------
Show evaluation reults
-------------------------------------------------------------------------"""
print('Test Error Rate = ' + str(1-accuracy))
print('Test Accuracy = '+ str(accuracy))
"""--------------------------------------------------------------------------
Show confusion matrix
-------------------------------------------------------------------------"""
#Draw a confusion matrix
predictions.groupBy("label","prediction").count().show()

=== start loading data ===
--2026-03-01 19:17:16--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/9-banking-data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 68020 (66K) [text/plain]
Saving to: ‘9-banking-data.csv’

9-banking-data.csv  100%[===================>]  66.43K  --.-KB/s    in 0.001s  

2026-03-01 19:17:17 (69.1 MB/s) - ‘9-banking-data.csv’ saved [68020/68020]

=== start clearning data ===
=== start basic statistical analysis on features ===
+-------+-------------------+------------------+-------------------+------------------+-------------------+-------------------+------------------+-------------------+--------------------+------------------+-------------------+
|summary|            OUTCOME|               AGE|          